In [1]:
import requests
import os
import time
import zipfile
import pandas as pd
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types

In [2]:
client = Minio(
    "minio:9000",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False
)
bucket = "crypto-data-lake"
if not client.bucket_exists(bucket):
    client.make_bucket(bucket)

In [3]:
spark = SparkSession.builder \
    .appName("LandingZone") \
    .config("spark.master", "spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.jars", ",".join([
        "/opt/spark-extra-jars/hadoop-aws-3.3.4.jar",
        "/opt/spark-extra-jars/aws-java-sdk-bundle-1.12.262.jar"
    ])) \
    .getOrCreate()

25/09/22 10:40:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
schema = types.StructType([
    types.StructField('agg_trade_id', types.LongType(), True), 
    types.StructField('price', types.DoubleType(), True), 
    types.StructField('quantity', types.DoubleType(), True), 
    types.StructField('first_trade_id', types.LongType(), True), 
    types.StructField('last_trade_id', types.LongType(), True), 
    types.StructField('timestamp', types.LongType(), True), 
    types.StructField('is_buyer_maker', types.BooleanType(), True), 
    types.StructField('is_best_match', types.BooleanType(), True)
])

In [6]:
extract_dir = "unzipped_data"
csv_file = os.path.join(extract_dir, os.listdir(extract_dir)[0])
print(f"Extracted CSV: {csv_file}")

Extracted CSV: unzipped_data/BTCUSDT-aggTrades-2025-08-01.csv


In [7]:
df = spark.read \
    .option("header", "false") \
    .schema(schema) \
    .csv(csv_file)

In [8]:
df.printSchema()

root
 |-- agg_trade_id: long (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: double (nullable = true)
 |-- first_trade_id: long (nullable = true)
 |-- last_trade_id: long (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- is_buyer_maker: boolean (nullable = true)
 |-- is_best_match: boolean (nullable = true)



In [9]:
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+------------+-----+--------+--------------+-------------+---------+--------------+-------------+
|agg_trade_id|price|quantity|first_trade_id|last_trade_id|timestamp|is_buyer_maker|is_best_match|
+------------+-----+--------+--------------+-------------+---------+--------------+-------------+
|           0|    0|       0|             0|            0|        0|             0|            0|
+------------+-----+--------+--------------+-------------+---------+--------------+-------------+



In [10]:
df.describe().show()

25/09/22 10:41:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 3:>                                                          (0 + 2) / 2]

+-------+-----------------+------------------+--------------------+-------------------+-------------------+--------------------+
|summary|     agg_trade_id|             price|            quantity|     first_trade_id|      last_trade_id|           timestamp|
+-------+-----------------+------------------+--------------------+-------------------+-------------------+--------------------+
|  count|          1314072|           1314072|             1314072|            1314072|            1314072|             1314072|
|   mean|   3.6411511565E9|114772.51094437456|0.018634520832961927|5.124889106730406E9|5.124889108896769E9|1.754049467280027...|
| stddev|379340.0558048148| 815.0145912458705| 0.13296616458618138| 1227424.3654440136| 1227424.8011467636|2.531215074523887...|
|    min|       3640494121|         112722.58|              1.0E-5|         5122977554|         5122977554|    1754006400328945|
|    max|       3641808192|          116052.0|             26.5822|         5127138382|         5

In [11]:
df.sample(0.001).show(10)

+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+
|agg_trade_id|    price|quantity|first_trade_id|last_trade_id|       timestamp|is_buyer_maker|is_best_match|
+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+
|  3640494896|115728.99|   0.011|    5122979049|   5122979049|1754006465067818|          true|         true|
|  3640495440|115738.75|    0.01|    5122980049|   5122980049|1754006503949238|         false|         true|
|  3640495758|115698.56|  8.0E-5|    5122980707|   5122980707|1754006524814845|          true|         true|
|  3640496100|115736.08|  6.9E-4|    5122981356|   5122981356|1754006551964157|          true|         true|
|  3640496118|115736.08| 0.01229|    5122981381|   5122981381|1754006553164578|          true|         true|
|  3640496174| 115739.7|  1.7E-4|    5122981474|   5122981474|1754006558556490|         false|         true|
|  3640497072|11577

In [12]:
df.withColumn("ingest_date", F.current_date()).withColumn("ingest_timestamp", F.current_timestamp()).show(truncate=False)

+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+-----------+--------------------------+
|agg_trade_id|price    |quantity|first_trade_id|last_trade_id|timestamp       |is_buyer_maker|is_best_match|ingest_date|ingest_timestamp          |
+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+-----------+--------------------------+
|3640494121  |115764.07|0.22677 |5122977554    |5122977554   |1754006400328945|true          |true         |2025-09-22 |2025-09-22 10:41:22.921545|
|3640494122  |115764.08|0.00145 |5122977555    |5122977555   |1754006400345714|false         |true         |2025-09-22 |2025-09-22 10:41:22.921545|
|3640494123  |115764.08|2.1E-4  |5122977556    |5122977556   |1754006400350235|false         |true         |2025-09-22 |2025-09-22 10:41:22.921545|
|3640494124  |115764.08|4.1E-4  |5122977557    |5122977557   |1754006400492405|false         |true         |2025

In [13]:
df = df.withColumn("ingest_date", F.current_date()) \
    .withColumn("ingest_timestamp", F.current_timestamp())

In [14]:
output_path = f"s3a://{bucket}/landing_zone/spot/daily/aggTrades/BTCUSDT/2025_08_01"
df.write.mode("overwrite").parquet(output_path)
print(f"Parquet written to: {output_path}")

25/09/22 10:41:36 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

Parquet written to: s3a://crypto-data-lake/landing_zone/spot/daily/aggTrades/BTCUSDT/2025_08_01


In [15]:
df = spark.read.parquet(output_path)

In [16]:
df.printSchema()

root
 |-- agg_trade_id: long (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: double (nullable = true)
 |-- first_trade_id: long (nullable = true)
 |-- last_trade_id: long (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- is_buyer_maker: boolean (nullable = true)
 |-- is_best_match: boolean (nullable = true)
 |-- ingest_date: date (nullable = true)
 |-- ingest_timestamp: timestamp (nullable = true)



In [17]:
df.describe().show()

+-------+-----------------+------------------+--------------------+-------------------+-------------------+--------------------+
|summary|     agg_trade_id|             price|            quantity|     first_trade_id|      last_trade_id|           timestamp|
+-------+-----------------+------------------+--------------------+-------------------+-------------------+--------------------+
|  count|          1314072|           1314072|             1314072|            1314072|            1314072|             1314072|
|   mean|   3.6411511565E9|114772.51094437456|0.018634520832961927|5.124889106730406E9|5.124889108896769E9|1.754049467280027...|
| stddev|379340.0558048148| 815.0145912458705| 0.13296616458618138| 1227424.3654440136| 1227424.8011467636|2.531215074523887...|
|    min|       3640494121|         112722.58|              1.0E-5|         5122977554|         5122977554|    1754006400328945|
|    max|       3641808192|          116052.0|             26.5822|         5127138382|         5

In [18]:
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+------------+-----+--------+--------------+-------------+---------+--------------+-------------+-----------+----------------+
|agg_trade_id|price|quantity|first_trade_id|last_trade_id|timestamp|is_buyer_maker|is_best_match|ingest_date|ingest_timestamp|
+------------+-----+--------+--------------+-------------+---------+--------------+-------------+-----------+----------------+
|           0|    0|       0|             0|            0|        0|             0|            0|          0|               0|
+------------+-----+--------+--------------+-------------+---------+--------------+-------------+-----------+----------------+



In [19]:
df.sample(0.001).show(truncate=False)

+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+-----------+--------------------------+
|agg_trade_id|price    |quantity|first_trade_id|last_trade_id|timestamp       |is_buyer_maker|is_best_match|ingest_date|ingest_timestamp          |
+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+-----------+--------------------------+
|3640495499  |115738.75|0.04974 |5122980116    |5122980116   |1754006514026974|false         |true         |2025-09-22 |2025-09-22 10:41:36.959815|
|3640496363  |115722.63|0.09388 |5122981966    |5122981989   |1754006572022774|true          |true         |2025-09-22 |2025-09-22 10:41:36.959815|
|3640499472  |115739.62|2.1E-4  |5122989088    |5122989088   |1754006823710199|false         |true         |2025-09-22 |2025-09-22 10:41:36.959815|
|3640500191  |115771.31|5.0E-5  |5122990747    |5122990747   |1754006898959976|false         |true         |2025